In [ ]:
from pathlib import Path

import torch
from IPython.display import Video

import hardware
from env import SKILLS, DuckEnv
from model import ActorCritic
from onpolicy import OnPolicy, fit, flat, normalize, sample_rollout
from rollout import evaluate, record_video

In [ ]:
hw = hardware.load("2026-11-07-hd1910")
env = DuckEnv(SKILLS["walk"], hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，和正式训练器一样
model = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")   # 61 个观测进，14 个关节的动作出

In [ ]:
def reward_to_go(rewards, dones, gamma):
    returns = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):              # 从最后一步往前算
        running = rewards[t] + gamma * running * (1 - dones[t])   # 回合在这一步结束就乘 0
        returns[t] = running
    return returns

In [ ]:
class Reinforce(OnPolicy):
    gamma = 0.99
    entropy_coef = 0.01

    def prepare(self, rollout):
        returns = reward_to_go(rollout["rewards"], rollout["dones"], self.gamma)
        return {**flat(rollout, "obs", "actions"), "advantages": normalize(returns).flatten()}

    def loss(self, batch):
        dist = self.model.action_distribution(batch["obs"])
        entropy = dist.entropy().sum(-1).mean()
        return self.policy_loss(dist, batch) - self.entropy_coef * entropy   # 减一点熵，鼓励探索

    def policy_loss(self, dist, batch):
        log_probs = dist.log_prob(batch["actions"]).sum(-1)      # 14 个关节的对数概率相加
        return -(log_probs * batch["advantages"]).mean()

In [ ]:
from model import compute_gae

In [ ]:
class A2C(Reinforce):
    lam = 0.95
    value_loss_coef = 1.0

    def prepare(self, rollout):
        # 超时只是截断，不是终点：把 critic 对那个状态的估值折扣后补进奖励
        bootstrap = self.gamma * rollout["values"].squeeze(-1) * rollout["time_outs"]
        rewards = rollout["rewards"] + bootstrap
        advantages, returns = compute_gae(rewards, rollout["dones"], rollout["values"],
                                          rollout["next_value"], self.gamma, self.lam)
        return {**flat(rollout, "obs", "critic_obs", "actions"),
                "advantages": normalize(advantages).flatten(),
                "returns": returns.flatten(0, 1)}

    def loss(self, batch):
        values = self.model.value(batch["critic_obs"])
        value_loss = self.value_loss(values, batch)
        return super().loss(batch) + self.value_loss_coef * value_loss

    def value_loss(self, values, batch):
        return (values - batch["returns"]).pow(2).mean()        # critic 的估值往价值目标靠

In [ ]:
def load(path):
    policy = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")
    policy.load_state_dict(torch.load(path))
    return policy


def compare(policies):
    counter = env.unwrapped.common_step_counter          # 都对齐到这一档课程表，同一个评测
    for name, policy in policies.items():
        result = evaluate(policy.act_inference, SKILLS["walk"], hw, counter)
        print(name, result["falls"], round(result["mean_step_reward"], 3))

In [ ]:
from torch.distributions import Normal, kl_divergence


class PPO(A2C):
    learning_rate = 1e-3
    clip_param = 0.2
    num_learning_epochs = 5
    num_mini_batches = 4
    desired_kl = 0.01

    def __init__(self, env, model, steps=24):
        super().__init__(env, model, steps)
        self.lr = self.learning_rate                     # 按 KL 调的当前学习率
        self.kl = torch.zeros(())                        # 最近一个小批里，新旧策略差了多少

    def prepare(self, rollout):
        # 再带上采样时的旧策略：算比值、裁价值、量 KL 都要用
        old = flat(rollout, "log_probs", "values", "action_means", "action_stds")
        return {**super().prepare(rollout), **old}

    def batches(self, rollout):
        data = self.prepare(rollout)
        size = len(data["actions"]) // self.num_mini_batches
        # 打乱一次，5 遍都用同一份划分
        order = torch.randperm(size * self.num_mini_batches, device=data["actions"].device)
        for _ in range(self.num_learning_epochs):
            for i in range(self.num_mini_batches):
                picked = order[i * size:(i + 1) * size]
                yield {key: value[picked] for key, value in data.items()}

    def policy_loss(self, dist, batch):
        with torch.no_grad():                            # 顺便量一下新旧策略差了多少
            old = Normal(batch["action_means"], batch["action_stds"])
            self.kl = kl_divergence(old, dist).sum(-1).mean()
        log_probs = dist.log_prob(batch["actions"]).sum(-1)
        ratio = torch.exp(log_probs - batch["log_probs"])           # 新旧概率之比
        clipped = ratio.clamp(1 - self.clip_param, 1 + self.clip_param)
        advantages = batch["advantages"]
        return -torch.min(ratio * advantages, clipped * advantages).mean()

    def value_loss(self, values, batch):
        old, returns = batch["values"], batch["returns"]
        clipped = old + (values - old).clamp(-self.clip_param, self.clip_param)
        return torch.max((values - returns).pow(2), (clipped - returns).pow(2)).mean()

    def on_before_optimizer_step(self, optimizer):
        kl = float(self.kl)
        if kl > 2 * self.desired_kl:                     # 这一步改得太多，学习率调小
            self.lr = max(1e-5, self.lr / 1.5)
        elif 0 < kl < self.desired_kl / 2:               # 改得太少，学习率调大
            self.lr = min(1e-2, self.lr * 1.5)
        for group in optimizer.param_groups:
            group["lr"] = self.lr

In [ ]:
fit(PPO(env, model), 1500)

In [ ]:
compare({"REINFORCE": load("../S03-policy-gradient/reinforce.pt"),
         "A2C": load("../S04-actor-critic/a2c.pt"), "PPO": model})

In [ ]:
counter = env.unwrapped.common_step_counter               # 课程表走到的那一档
record_video(model.act_inference, SKILLS["walk"], hw, counter, Path("ppo.mp4"))
Video("ppo.mp4", embed=True)

In [ ]:
torch.save(model.state_dict(), "ppo.pt")